In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset, random_split
import pandas as pd
import numpy as np
import torch

In [ ]:
df = pd.read_csv('dataset.tsv', sep='\t', names=['text','phrase1','phrase2','label'])
df

In [ ]:
label_to_index_map = {'not-related': 0, 'related': 1}
index_to_label_map = {0: 'not-related', 1: 'related'}

df['labels'] = df['label'].map(label_to_index_map)

In [ ]:
special_tokens = {
    'additional_special_tokens': ['[BOTEXT]', '[EOTEXT]', '[BOPHRASE]', '[EOPHRASE]']
}

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
num_added = tokenizer.add_special_tokens(special_tokens)
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

In [ ]:
def format_input(text, phrase1, phrase2):
    marked = text.replace(phrase1, f"[BOPHRASE] {phrase1} [EOPHRASE]", 1)
    marked = marked.replace(phrase2, f"[BOPHRASE] {phrase2} [EOPHRASE]", 1)
    return f"[BOTEXT] {marked} [EOTEXT]"

In [ ]:
df['input_text'] = df.apply(
    lambda row: format_input(row['text'], row['phrase1'], row['phrase2']), axis=1
)
inputs = df['input_text'].tolist()
labels = df['labels'].tolist()

inputs[0]

In [ ]:
class RelationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]).to(device) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]).to(device)
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
dataset = RelationDataset(inputs, labels, tokenizer)
total_size = len(dataset)
val_size = int(0.1 * total_size)
train_size = total_size - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    logging_dir='./logs'
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)


In [ ]:
train_result = trainer.train()

print("Fine-tuning complete. Model and tokenizer saved to ./relation_classifier_bert")

In [ ]:
preds_output = trainer.predict(val_dataset)
correct = (np.argmax(preds_output.predictions, axis=1) == preds_output.label_ids).sum()
total = len(preds_output.label_ids)
accuracy = correct / total
print(f"  Correct predictions: {correct} / {total}")
print(f"  Accuracy: {accuracy:.4f}")

In [ ]:
val_indices = val_dataset.indices if hasattr(val_dataset, 'indices') else val_dataset.indices

for i, idx in enumerate(val_indices):
    if np.argmax(preds_output.predictions, axis=1)[i] != preds_output.label_ids[i]:
        text = df.loc[idx, 'text']
        phrase1 = df.loc[idx, 'phrase1']
        phrase2 = df.loc[idx, 'phrase2']
        true_label = df.loc[idx, 'label']
        pred_label = index_to_label_map[int(np.argmax(preds_output.predictions, axis=1)[i])]
        print(f"Text: {text}")
        print(f"  Phrase1: {phrase1}")
        print(f"  Phrase2: {phrase2}")
        print(f"  True: {true_label}, Predicted: {pred_label}")


In [ ]:
model.save_pretrained('./relation_classifier_bert')
tokenizer.save_pretrained('./relation_classifier_bert')